<a href="https://colab.research.google.com/github/Giocrisrai/taller-ia-agentes-biblioteca/blob/main/notebooks/03_Tarea_Plantilla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

> **Ábrelo en Colab con el botón de arriba.** Una vez dentro, menú `Archivo` →
> `Guardar una copia en Drive`, para tener tu propia versión editable.


# Tarea evaluada — Tu propio agente de IA

**DUOC UC · Bibliotecas** · Curso de IA Aplicando Agentes

---

## ✍️ Completa esto antes de empezar

Haz **doble clic sobre esta celda** para editarla, escribe tus datos y presiona
`Shift + Enter` para volver a verla formateada.

| | |
|---|---|
| **Nombre** | *(escribe aquí)* |
| **Carrera** | *(escribe aquí)* |
| **Dominio de mi agente** | *(escribe aquí: cafetería, gimnasio, secretaría, taller mecánico...)* |

---

## El camino: 10 pasos

Los pasos 1 a 3 son preparación y no se evalúan. Del 4 al 8 está lo que puntúa.

| | Paso | Qué haces | Puntos |
|---|---|---|---|
| 🔑 | **1** | Preparas el entorno | — |
| 💭 | **2** | Eliges tu dominio y sus datos | — |
| 🔧 | **3** | Armas tu agente | — |
| 🔧 | **4** | Escribes tus **3 herramientas** | **20** |
| 🤖 | **5** | Una consulta que **encadene 2** | **20** |
| 🧠 | **6** | **3 turnos con memoria** | **20** |
| 🛡️ | **7** | Un **límite** que se vea actuando | **20** |
| ✍️ | **8** | Un párrafo sobre **un riesgo** | **20** |
| ✅ | **9** | Repasas la lista de verificación | — |
| 📤 | **10** | Compartes el enlace | — |

**Total: 100 puntos**, que son el 100% de la nota del curso. Se aprueba con 60 (nota 4,0).

📅 **Fecha de entrega: lunes 14 de septiembre de 2026, 23:00.**
La rúbrica completa está publicada junto a la actividad en el campus.

> ⚠️ **Hay que entregarlo ejecutado.** Un notebook sin resultados visibles no se puede
> evaluar. El Paso 9 te lo recuerda.


---
---

# 🔑 Paso 1 · Prepara el entorno

⏱️ *3 minutos* · Tres celdas seguidas. Ejecútalas en orden y sigue.

Es exactamente lo mismo de las dos sesiones del taller. Si tu llave ya está guardada en
los Secrets de Colab, solo hay que ejecutar.

### Si te falta la llave

**1.a** [console.groq.com](https://console.groq.com/) → **API Keys** → **Create API Key** → cópiala.

**1.b** Aquí en Colab: barra lateral izquierda → ícono de **llave** → **Add new secret** →
**Name**: `GROQ_API_KEY` → **Value**: tu llave.

**1.c** ⚠️ Activa el interruptor **Notebook access**.

> ⚠️ **La tarea se hace con la llave de verdad.** El modo simulado no muestra al agente
> razonando, y sin esas trazas los requisitos de los Pasos 5, 6 y 7 no se pueden evaluar.

> 🤔 **«¿Por qué dice `openai` si estamos usando Groq?»** Es la pregunta que siempre
> sale, y la respuesta es que **no estás usando OpenAI en ningún momento**:
>
> | Dónde lo ves | Qué es de verdad |
> |---|---|
> | `openai/gpt-oss-120b` | El **nombre del modelo**. GPT-OSS son unos pesos abiertos que OpenAI publicó, y **Groq los hospeda y ejecuta**. Se cobra a tu cupo de Groq. |
> | `create_openai_tools_agent` | El **nombre de una función de LangChain**, por el formato de llamada a herramientas que popularizó OpenAI y que Groq también implementa. |
>
> No necesitas cuenta de OpenAI ni `OPENAI_API_KEY`. **Todo el taller funciona con tu
> única llave de Groq.**

> ⚠️ **La primera vez, Colab te va a mostrar un aviso.** Dice
> *«Warning: This notebook was not authored by Google»* y aparece porque el notebook viene
> de GitHub. **Es normal.** Haz clic en **`Run anyway`** (Ejecutar de todos modos) y sigue.


In [ ]:
# ▶ PASO 1a · Instala las librerías.
import sys

if "google.colab" in sys.modules:
    %pip install -qU langchain langchain-classic langchain-groq groq
    print("✅ Librerías instaladas.")
else:
    print("✅ Entorno local detectado.")


In [ ]:
# ▶ PASO 1b · Carga tu llave.
import os

try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY") or ""
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

MODELO = "openai/gpt-oss-20b"
MODO_SIMULADO = not os.getenv("GROQ_API_KEY")

if MODO_SIMULADO:
    print("⚠️  Sin llave: MODO SIMULADO.")
    print("    Para la ENTREGA necesitas la llave de verdad: en modo simulado no se ve")
    print("    al agente razonando y los Pasos 5, 6 y 7 no se pueden evaluar.")
    print("    Revisa los puntos 1.a a 1.c de arriba.")
else:
    print(f"✅ Paso 1 listo. Llave cargada. Modelo: {MODELO}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ▶ PASO 1c · Celda técnica: lo mismo del taller, reunido aquí.
# No necesitas leerla ni modificarla. Ejecútala y sigue al Paso 2.
# ─────────────────────────────────────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.agents import AgentExecutor, create_openai_tools_agent


class AgenteSimulado:
    """Agente de emergencia que funciona SIN modelo de lenguaje.

    No razona: elige la herramienta comparando palabras de la pregunta con el
    nombre y la descripción de cada una, y adivina los parámetros con reglas
    fijas. Es aproximado a propósito. Sirve para que puedas seguir la clase si
    no lograste tu llave, no para aprender cómo decide un agente de verdad.
    """

    import re as _re

    DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

    def __init__(self, herramientas):
        self.herramientas = herramientas

    def _campos(self, herramienta):
        """Nombres de los parámetros que la herramienta necesita."""
        try:
            return list(herramienta.args.keys())
        except Exception:
            return []

    def _adivinar(self, campo, pregunta):
        """Intenta sacar el valor de un parámetro con reglas simples."""
        c = campo.lower()
        if "hora" in c:
            m = self._re.search(r"\b(\d{1,2}:\d{2})\b", pregunta)
            return m.group(1) if m else None
        if "sala" in c:
            m = self._re.search(r"[Ss]ala\s*(\d+)", pregunta)
            return f"Sala {m.group(1)}" if m else None
        if "codigo" in c or "código" in c:
            m = self._re.search(r"\b([A-Za-z]-?\d{3})\b", pregunta)
            return m.group(1).upper() if m else None
        if "dia" in c or "día" in c:
            for d in self.DIAS:
                if d in pregunta.lower():
                    return d
            return None
        if "titulo" in c or "título" in c or "libro" in c:
            m = self._re.search(r"libro\s+(?:llamado\s+|titulado\s+)?([A-ZÁÉÍÓÚÑ]\w*(?:\s+[A-ZÁÉÍÓÚÑ]\w*)*)", pregunta)
            return m.group(1) if m else None
        return None

    def invoke(self, entrada):
        pregunta = entrada.get("input", "")
        historial = entrada.get("chat_history", [])
        palabras = set(pregunta.lower().replace("¿", " ").replace("?", " ").split())

        mejor, top = None, 0
        for h in self.herramientas:
            texto = (h.name + " " + (h.description or "")).lower()
            p = sum(1 for w in palabras if len(w) > 3 and w in texto)
            if p > top:
                mejor, top = h, p

        print("> Entrando en la cadena del agente...  [MODO SIMULADO]")
        if historial:
            print(f"  (recibe {len(historial)} mensajes de lo ya conversado)")

        if mejor is None:
            print("Pensamiento: ninguna herramienta encaja con la pregunta.")
            salida = "(modo simulado) No tengo una herramienta para responder eso."
            print("> Cadena terminada.")
            return {"output": salida}

        print(f"Pensamiento: esto parece requerir la herramienta '{mejor.name}'.")

        campos = self._campos(mejor)
        argumentos, faltan = {}, []
        for campo in campos:
            valor = self._adivinar(campo, pregunta)
            if valor is None:
                faltan.append(campo)
            else:
                argumentos[campo] = valor

        if faltan:
            print(f"Acción: {mejor.name}  ← no pude deducir: {', '.join(faltan)}")
            print("   (el modo simulado adivina los parámetros con reglas fijas.")
            print("    Un agente real los deduce leyendo la frase completa.)")
            salida = f"(modo simulado) Necesitaría el valor de: {', '.join(faltan)}."
            print("> Cadena terminada.")
            return {"output": salida}

        print(f"Acción: {mejor.name}({argumentos})")
        try:
            observacion = mejor.invoke(argumentos)
        except Exception as e:
            observacion = f"(no pude ejecutarla: {type(e).__name__})"
        print(f"Observación: {observacion}")
        print("> Cadena terminada.")
        return {"output": f"(modo simulado) {observacion}"}


def armar_agente(herramientas, instrucciones, max_pasos=6):
    """Une modelo + herramientas + instrucciones y devuelve un agente listo."""
    if MODO_SIMULADO:
        return AgenteSimulado(herramientas)
    prompt = ChatPromptTemplate.from_messages([
        ("system", instrucciones),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ])
    llm = ChatGroq(model=MODELO, temperature=0)
    cerebro = create_openai_tools_agent(llm, herramientas, prompt)
    return AgentExecutor(agent=cerebro, tools=herramientas, verbose=True,
                         max_iterations=max_pasos, handle_parsing_errors=True)


def preguntar(agente, mensaje, historial=None):
    """Envía un mensaje al agente. Nunca lanza excepción."""
    try:
        return agente.invoke({"input": mensaje, "chat_history": historial or []})["output"]
    except Exception as e:
        return f"❌ {type(e).__name__}: {e}\n   Si dice 'tool_use_failed', vuelve a ejecutar la celda."


def conversar(agente, mensaje, historial):
    """Como preguntar(), pero además guarda el turno en el historial (memoria)."""
    respuesta = preguntar(agente, mensaje, historial)
    historial.append(HumanMessage(content=mensaje))
    historial.append(AIMessage(content=respuesta))
    return respuesta


print("✅ Paso 1 completo. Sigue con el Paso 2.")


---
---

# 💭 Paso 2 · Elige tu dominio y escribe sus datos

⏱️ *10 minutos*

Tu agente necesita "algo que consultar". **No hace falta una base de datos**: unos
diccionarios de Python bastan, igual que las `SALAS` del taller del lunes.

Y si te animas, puede consultar una **API real**, igual que `buscar_libro` consultaba
Open Library. Más abajo te dejo cuatro gratuitas y sin llave.

### Elige algo que conozcas

| Dominio | Herramientas que podría tener |
|---|---|
| Cafetería del campus | consultar precio, ver horario, registrar pedido |
| Gimnasio | ver clases del día, consultar cupos, inscribir a una clase |
| Secretaría de tu carrera | buscar requisitos de un trámite, consultar plazos, agendar hora |
| Taller mecánico | consultar precio de servicio, ver disponibilidad, agendar |
| Tu emprendimiento | lo que tú vendas |

**Los datos pueden ser inventados**, y con eso apruebas: lo que se evalúa es el diseño
del agente. Pero si quieres que tu agente consulte el mundo real, más abajo te dejo
cuatro APIs gratuitas y sin llave.

### Qué hacer en la celda

1. **Borra el ejemplo de la cafetería**
2. Escribe **tus propios** diccionarios
3. Ejecuta y comprueba que imprime lo que esperas

---

### 🌍 Sube el nivel: usa datos reales

Los datos inventados sirven y **con eso apruebas**. Pero una herramienta es una función de
Python, así que puede llamar a **cualquier API del mundo**. Estas son gratuitas y **no
piden llave** — funcionan copiando la URL:

| API | Qué devuelve | URL de ejemplo |
|---|---|---|
| **Open Library** | Libros reales: autor, año, ediciones | `https://openlibrary.org/search.json?q=sapiens&limit=3` |
| **Wikipedia ES** | Resumen de cualquier tema | `https://es.wikipedia.org/api/rest_v1/page/summary/Chile` |
| **mindicador.cl** | Dólar, UF y UTM de hoy en Chile | `https://mindicador.cl/api` |
| **Feriados de Chile** | Los feriados del año | `https://api.boostr.cl/holidays.json` |

El patrón es el mismo que viste en el taller:

```python
import json, urllib.request

@tool
def valor_del_dolar() -> str:
    """Consulta a cuánto está el dólar hoy en Chile. Úsala cuando pregunten
    por el tipo de cambio o el precio del dólar."""
    try:
        with urllib.request.urlopen("https://mindicador.cl/api", timeout=10) as r:
            d = json.loads(r.read())
        return f"Hoy el dólar está a ${d['dolar']['valor']} y la UF a ${d['uf']['valor']}."
    except Exception as e:
        return f"No pude consultar el indicador ahora ({type(e).__name__})."
```

> ⚠️ **Envuelve siempre la llamada en `try / except`.** Las APIs de verdad fallan a veces,
> y una herramienta que revienta deja al agente sin respuesta. Si devuelves un mensaje
> claro, el agente incluso puede reintentar solo.

**Esto no es obligatorio, pero cuenta.** Un agente que consulta datos reales demuestra que
entendiste lo esencial: que el agente no sabe de dónde vienen los datos, y que eso lo
decides tú.


In [ ]:
# ▶ PASO 2 · Tus datos.
# ── EJEMPLO (bórralo y pon lo tuyo) ──────────────────────────────────────────
# Dominio de ejemplo: una cafetería.

PRODUCTOS = {
    "café":      {"precio": 1500, "stock": 40},
    "sándwich":  {"precio": 3200, "stock": 6},
    "brownie":   {"precio": 1800, "stock": 0},
}

PEDIDOS = []

print("✅ Paso 2 listo. Datos cargados:", list(PRODUCTOS))


---
---

# 🔧 Paso 3 · Escribe las instrucciones de tu agente

⏱️ *3 minutos*

Quién es tu agente, cómo responde y qué tono usa. Una o dos frases bastan.

Los límites vienen después, en el Paso 7: aquí solo la identidad.


In [ ]:
# ▶ PASO 3 · Las instrucciones de tu agente.

MIS_INSTRUCCIONES = (
    "Eres el asistente de ___________. "          # ← completa con tu dominio
    "Responde en español, breve y amable. "
    "Si tienes una herramienta que puede darte el dato, úsala en vez de suponerlo."
)

print("✅ Paso 3 listo.")


---
---

# 🔧 Paso 4 · Tus tres herramientas  ·  20 puntos

⏱️ *20 minutos* · **Este es el primer paso que puntúa.**

Escribe **tres** funciones con `@tool`. Recuerda las dos reglas del taller:

1. **La descripción entre comillas triples es lo único que lee el modelo.** Tiene que
   decir *cuándo* usar la herramienta, no solo qué hace
2. Si la herramienta **cambia algo** (registra, reserva, cancela), **valida los datos
   dentro de la función**. El modelo puede alucinar; tu código no

Al menos una de las tres debería cambiar el estado, para que el Paso 7 tenga sentido.

### Cómo se evalúa este paso

| Puntos | Qué tiene que verse |
|:--:|---|
| **20** | Tres tools propias, descripciones que dicen **cuándo** usarlas, y al menos una **valida** sus datos |
| 15 | Tres tools correctas, pero alguna descripción es genérica o falta la validación |
| 10 | Las descripciones no permitirían al modelo elegir bien |
| 5 | Menos de tres, o copiadas del taller con el nombre cambiado |


In [ ]:
# ▶ PASO 4 · Tus tres herramientas.

# ── EJEMPLO: herramienta 1 de 3 (solo consulta) ──────────────────────────────
@tool
def consultar_precio(producto: str) -> str:
    """Consulta el precio y la disponibilidad de un producto de la cafetería.
    Recibe el nombre del producto. Úsala cuando pregunten cuánto cuesta algo
    o si queda disponible."""
    clave = producto.strip().lower()
    if clave not in PRODUCTOS:
        return f"No vendemos '{producto}'. Tenemos: {', '.join(PRODUCTOS)}."
    p = PRODUCTOS[clave]
    if p["stock"] == 0:
        return f"El {clave} cuesta ${p['precio']} pero está AGOTADO."
    return f"El {clave} cuesta ${p['precio']} y quedan {p['stock']} unidades."


# ── TU HERRAMIENTA 2 ─────────────────────────────────────────────────────────
# @tool
# def ...


# ── TU HERRAMIENTA 3 ─────────────────────────────────────────────────────────
# Esta debería CAMBIAR algo (registrar, reservar, cancelar) y validar los datos.
# @tool
# def ...


mis_herramientas = [consultar_precio]   # ← agrega aquí las tuyas

mi_agente = armar_agente(mis_herramientas, MIS_INSTRUCCIONES)

print(f"✅ Paso 4: tengo {len(mis_herramientas)} herramientas de 3:")
for h in mis_herramientas:
    print("   ·", h.name)
if len(mis_herramientas) < 3:
    print("\n⚠️  Te faltan herramientas para cumplir el requisito.")


---

# 🤖 Paso 5 · Una consulta que encadene dos  ·  20 puntos

⏱️ *5 minutos*

Escribe una pregunta que **obligue** al agente a usar dos herramientas distintas, una
detrás de otra.

> 💡 **Truco:** pide dos cosas en la misma frase, como el *"necesito Sapiens y una sala
> a las 16:00"* del taller.

### Cómo se evalúa este paso

**En la traza tienen que verse dos `Invoking:` con nombres distintos.** Si aparece solo
uno, o el mismo repetido, no cumple.


In [ ]:
# ▶ PASO 5 · Tu consulta encadenada.

consulta_encadenada = "___________"   # ← escribe aquí tu consulta

print(preguntar(mi_agente, consulta_encadenada))


---

# 🧠 Paso 6 · Tres turnos con memoria  ·  20 puntos

⏱️ *8 minutos* · Dos celdas: la conversación y la comprobación.

**El turno 3 tiene que depender de algo dicho en el turno 1**, para demostrar que la
memoria funciona de verdad.

Ejemplo de la estructura que se busca:

1. *"Hola, soy Camila. ¿Cuánto cuesta el café?"*
2. *"¿Y el sándwich?"*
3. *"¿Cómo me llamo y qué fue lo primero que pregunté?"* ← solo se responde con memoria

### Cómo se evalúa este paso

| Puntos | Qué tiene que verse |
|:--:|---|
| **20** | El turno 3 es **imposible** de responder sin el turno 1, y el agente lo responde bien |
| 15 | El turno 3 depende del turno 2, no del 1 |
| 10 | Tres turnos, pero ninguno demuestra que se usó la memoria |
| 5 | Menos de tres turnos, o no se pasa el historial |

> ⚠️ Un turno 3 del tipo *"gracias, adiós"* **no demuestra memoria y no puntúa**.


In [ ]:
# ▶ PASO 6 · Tu conversación de tres turnos.

historial = []   # ← la memoria: una lista de mensajes

print("TURNO 1")
print(conversar(mi_agente, "___________", historial))

print()
print("TURNO 2")
print(conversar(mi_agente, "___________", historial))

print()
print("TURNO 3  (este debe depender del turno 1)")
print(conversar(mi_agente, "___________", historial))


In [ ]:
# ▶ PASO 6 · Comprobación: la memoria por dentro. Debe tener 6 mensajes.

print(f"El historial tiene {len(historial)} mensajes:\n")
for i, m in enumerate(historial, 1):
    quien = "PERSONA" if isinstance(m, HumanMessage) else "AGENTE "
    print(f"{i}. [{quien}] {m.content[:90]}")

if len(historial) != 6:
    print("\n⚠️  Deberían ser 6 mensajes (3 turnos). Revisa la celda anterior.")


---

# 🛡️ Paso 7 · Un límite que se vea actuando  ·  20 puntos

⏱️ *8 minutos*

Elige **al menos uno** de los tres tipos que vimos:

| Tipo | Cómo se hace |
|---|---|
| Instrucción | Una regla en `MIS_INSTRUCCIONES` (*"nunca hagas X sin confirmación"*) |
| Tope de pasos | `max_pasos=3` al llamar a `armar_agente` |
| Validación en el código | Un `if` dentro de la herramienta que rechaza datos inválidos |

### Cómo se evalúa este paso

**No basta con escribirlo: hay que verlo actuar.** Provoca la situación en la que el
límite tiene que saltar, y que se vea en el resultado.

| Puntos | Qué tiene que verse |
|:--:|---|
| **20** | Se provoca la situación y el resultado muestra que el agente **no hizo** lo que no debía |
| 15 | Se provoca, pero el resultado es ambiguo o no se comprueba el efecto |
| 10 | El límite está declarado pero **nunca se provoca** la situación |
| 5 | Se menciona la idea sin implementarla |

> 💡 La mejor evidencia es imprimir el estado antes y después (como el `RESERVAS` del taller).


In [ ]:
# ▶ PASO 7 · Tu límite, y la situación que lo hace saltar.

INSTRUCCIONES_CON_LIMITE = (
    MIS_INSTRUCCIONES +
    " REGLA CRÍTICA: ___________"      # ← escribe tu regla
)

agente_con_limite = armar_agente(mis_herramientas, INSTRUCCIONES_CON_LIMITE, max_pasos=4)

# ← Provoca aquí la situación en la que el límite tiene que actuar
print(preguntar(agente_con_limite, "___________"))

# ← Imprime aquí el estado, para demostrar que el límite funcionó
# print("Estado después:", PEDIDOS or "ninguno")


---

# ✍️ Paso 8 · Un riesgo de tu agente  ·  20 puntos

⏱️ *10 minutos*

**Escribe tu respuesta en esta misma celda de texto.** Haz doble clic para editarla,
escribe debajo de "Mi respuesta", y presiona `Shift + Enter` cuando termines.

Máximo **150 palabras**. Responde estas dos cosas:

1. **¿Qué puede salir mal con TU agente concreto?** No vale *"puede alucinar"* en general:
   di qué pasaría en tu dominio y a quién le afectaría
2. **¿Cómo lo reducirías?** Una medida concreta: una validación, una confirmación,
   una herramienta que no le darías

### Cómo se evalúa este paso

| Puntos | Qué tiene que verse |
|:--:|---|
| **20** | Riesgo **específico de tu agente**, con el daño concreto y a quién afecta, y una mitigación implementable |
| 15 | Riesgo específico, pero la mitigación es vaga (*"supervisar más"*) |
| 10 | Riesgo genérico aplicado superficialmente a tu dominio |
| 5 | *"Puede alucinar"* sin conexión con tu agente |

> 💡 **Pregúntate:** ¿este párrafo se podría copiar y pegar en la entrega de cualquier
> otro compañero? Si sí, es genérico.
>
> 💡 Si tu agente **falló** durante las pruebas y lo documentas honestamente aquí, **eso suma**.

---

### ✍️ Mi respuesta

*(escribe aquí)*

---


---
---

# ✅ Paso 9 · Repasa antes de entregar

⏱️ *5 minutos*

Cada punto de esta lista es una línea de la rúbrica. Repásalos uno a uno.

**Contenido**

- [ ] Completé mi nombre, carrera y dominio en la primera celda
- [ ] **Paso 4** — tengo **3 herramientas** con `@tool` y descripciones que dicen *cuándo* usarlas
- [ ] **Paso 4** — al menos una herramienta **valida** los datos que recibe
- [ ] **Paso 5** — la consulta muestra **dos `Invoking:` distintos** en la traza
- [ ] **Paso 6** — los 3 turnos funcionan y el turno 3 **depende del turno 1**
- [ ] **Paso 6** — la comprobación dice **6 mensajes**
- [ ] **Paso 7** — el límite **se ve actuando**, no solo escrito
- [ ] **Paso 8** — escribí el párrafo (máx. 150 palabras) y es específico de mi agente

**Ejecución** ← lo que más se falla

- [ ] Ejecuté el notebook **entero, de arriba abajo**, en orden
- [ ] Se ven **todos** los resultados debajo de cada celda
- [ ] No quedó ningún `___________` sin reemplazar
- [ ] No estoy en modo simulado

> 💡 Para asegurarte: menú **`Entorno de ejecución`** → **`Reiniciar y ejecutar todo`**.
> Así compruebas que funciona de principio a fin sin depender del orden en que lo fuiste
> probando. Espera a que terminen todas las celdas antes de entregar.


---

# 📤 Paso 10 · Comparte el enlace

⏱️ *2 minutos*

**10.1** Menú → **`Archivo`** → **`Guardar una copia en Drive`** (si no lo hiciste ya)

**10.2** Botón **`Compartir`**, arriba a la derecha

**10.3** En *Acceso general*, cambia a **`Cualquier persona con el enlace`**

**10.4** Deja el rol en **`Lector`**

**10.5** **`Copiar vínculo`** y entrégalo donde indicó el docente

---

### Comprueba que el enlace funciona

Ábrelo en una **ventana de incógnito** (`Cmd + Shift + N` en Mac, `Ctrl + Shift + N` en
Windows). Si se ve el notebook con todos los resultados, está listo.

**Si pide permiso, el paso 10.3 no quedó bien hecho.** Vuelve y repítelo.

---

> 🙋 Si te atascaste en algo, **entrega igual lo que alcanzaste**. Un notebook incompleto
> pero ejecutado vale mucho más que uno perfecto sin ejecutar.
